# Datos

Descarga de Amazon Reviews 2018 y preprocesamiento con el código de LatentR3 (`R3/data/process.py`).

## Dependencias

In [ ]:
!conda env create -n your_env -f R3/environment.yaml

Este virtual enviroment utilizado depende de librerías compiladas para una GPU de NVIDIA. Esto puede complicar el entrenamiento del LatentR3 para algunos integrantes que tiene solo una AMD Radeon. Se considerará para la siguiente entrega e investigarán alternativas.

## Código de los autores

In [ ]:
import os, sys, gzip, json, time, datetime, glob, urllib.request
import pandas as pd

ROOT = os.path.abspath(".")
RAW = os.path.join(ROOT, "raw")
REPO = os.path.join(ROOT, "repo_latentr3")
CATEGORIES = ["Musical_Instruments", "Video_Games", "CDs_and_Vinyl", "Toys_and_Games"]

if not os.path.exists(REPO):
    !git clone https://github.com/xuwenxinedu/R3 "{REPO}"
sys.path.insert(0, os.path.join(REPO, "data"))
from process import gao

## Descarga

In [ ]:
BASE = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_v2"
os.makedirs(RAW, exist_ok=True)
for cat in CATEGORIES:
    for url in [f"{BASE}/categoryFiles/{cat}.json.gz", f"{BASE}/metaFiles2/meta_{cat}.json.gz"]:
        path = os.path.join(RAW, os.path.basename(url))
        if not os.path.exists(path):
            print("Descargando", url)
            urllib.request.urlretrieve(url, path + ".part")
            os.rename(path + ".part", path)

## Preprocesamiento

Se fija la zona horaria en UTC porque `gao` usa la hora local para las ventanas de tiempo. Solo se cargan reseñas desde 2010 y los campos que usa `gao`.

In [6]:
os.environ["TZ"] = "UTC"
time.tzset()
os.chdir(REPO)
since = datetime.datetime(2010, 1, 1).timestamp()

for cat in CATEGORIES:
    reviews = []
    with gzip.open(os.path.join(RAW, f"{cat}.json.gz"), "rt") as f:
        for line in f:
            r = json.loads(line)
            if r["unixReviewTime"] >= since:
                reviews.append({k: r[k] for k in ["reviewerID", "asin", "overall", "unixReviewTime"]})
    metadata = []
    with gzip.open(os.path.join(RAW, f"meta_{cat}.json.gz"), "rt") as f:
        for line in f:
            m = json.loads(line)
            metadata.append({k: m[k] for k in ["asin", "title"] if k in m})
    gao(cat, metadata=metadata, reviews=reviews)
    del reviews, metadata

2026-10-09 01:53:43.612 | INFO     | process:gao:20 - from 1506816000 to 1541030400
2026-10-09 01:53:43.612 | INFO     | process:gao:33 - from Musical_Instruments metadata: 120310 reviews: 1482767
100%|██████████| 1482767/1482767 [00:00<00:00, 6300145.02it/s]
2026-10-09 01:53:43.850 | INFO     | process:gao:45 - users: 139149, items: 32870, reviews: 1482767, density: 0.00032418515080770546
100%|██████████| 1482767/1482767 [00:00<00:00, 5724983.53it/s]
2026-10-09 01:53:44.404 | INFO     | process:gao:96 - users: 119304, items: 30544, reviews: 160201, density: 4.3962694910382435e-05
100%|██████████| 1482767/1482767 [00:00<00:00, 6326968.11it/s]
2026-10-09 01:53:44.642 | INFO     | process:gao:96 - users: 2270, items: 3608, reviews: 11645, density: 0.0014218281449935044
2026-10-09 01:53:44.643 | INFO     | process:gao:20 - from 1498867200 to 1541030400
2026-10-09 01:53:44.643 | INFO     | process:gao:33 - from Musical_Instruments metadata: 120310 reviews: 1482767
100%|██████████| 1482767/